In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [2]:
# Generate a synthetic dataset: 5000 rows, 20 columns
rng = np.random.RandomState(42)
n_rows = 5000

numeric_cols = [f"num_{i}" for i in range(1, 16)]   # 15 numeric features
cat_cols = [f"cat_{i}" for i in range(1, 6)]         # 5 categorical features

data = {}
for col in numeric_cols:
    data[col] = rng.normal(loc=rng.uniform(10, 100), scale=rng.uniform(5, 25), size=n_rows)

for col in cat_cols:
    n_levels = rng.randint(3, 6)
    levels = [f"level_{j}" for j in range(n_levels)]
    data[col] = rng.choice(levels, size=n_rows)

df = pd.DataFrame(data)

# Inject missing values into a few numeric columns
for col in rng.choice(numeric_cols, size=4, replace=False):
    missing_idx = rng.choice(df.index, size=int(0.05 * n_rows), replace=False)
    df.loc[missing_idx, col] = np.nan

# Build an imbalanced binary target from a weighted combination of numeric features
weights = rng.uniform(-3, 3, size=len(numeric_cols))
score = df[numeric_cols].fillna(df[numeric_cols].mean()).values @ weights
threshold = np.percentile(score, 90)   # ~90/10 class split
df["target"] = (score > threshold).astype(int)

df["target"].value_counts()


target
0    4500
1     500
Name: count, dtype: int64

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("reducer", PCA(n_components=10)),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


In [5]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1 score:  {f1:.3f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy:  0.795
Precision: 0.296
Recall:    0.760
F1 score:  0.426
Confusion matrix:
 [[719 181]
 [ 24  76]]


In [6]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="f1")
print("CV F1:", cv_scores.mean())


CV F1: 0.4601779111841031


In [7]:
param_grid = {"model__C": [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(X_train, y_train)

print("Best C:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)


Best C: {'model__C': 1}
Best CV F1: 0.4601779111841031


In [8]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Baseline -> F1: {:.3f}, Accuracy: {:.3f}".format(f1, acc))
print("Tuned    -> F1: {:.3f}, Accuracy: {:.3f}".format(
    f1_score(y_test, y_pred_best),
    accuracy_score(y_test, y_pred_best)
))


Baseline -> F1: 0.426, Accuracy: 0.795
Tuned    -> F1: 0.426, Accuracy: 0.795
